In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Load the datasets
train_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/reservation_cancellation/train.csv'
test_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/reservation_cancellation/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Copy the DataFrames to avoid modifying the original data
train_df_copy = train_df.copy()
test_df_copy = test_df.copy()

# Identify categorical columns
categorical_columns = train_df_copy.select_dtypes(include=['object']).columns

# Apply Label Encoding to categorical columns
label_encoder = LabelEncoder()
for col in categorical_columns:
    train_df_copy[col] = label_encoder.fit_transform(train_df_copy[col])
    test_df_copy[col] = label_encoder.transform(test_df_copy[col])

# Display the first few rows of the processed datasets
train_df_copy.head(), test_df_copy.head()


(      id  no_of_adults  ...  no_of_special_requests  booking_status
 0  15559             2  ...                       2               0
 1  32783             2  ...                       1               0
 2  11797             3  ...                       0               1
 3  39750             2  ...                       1               1
 4  28711             2  ...                       0               1
 
 [5 rows x 19 columns],
       id  no_of_adults  ...  no_of_special_requests  booking_status
 0   8768             2  ...                       1               0
 1  38340             2  ...                       0               1
 2   7104             2  ...                       0               0
 3  36898             2  ...                       3               0
 4   9747             2  ...                       1               0
 
 [5 rows x 19 columns])

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Print column information for the transformed training dataset
column_info_train = get_column_info(train_df_copy)
print("Column information for the transformed training dataset:")
print(column_info_train)

# Print column information for the transformed test dataset
column_info_test = get_column_info(test_df_copy)
print("Column information for the transformed test dataset:")
print(column_info_test)


2025-08-31 12:06:27.559 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


Column information for the transformed training dataset:
{'Category': [], 'Numeric': ['id', 'no_of_adults', 'no_of_children', 'no_of_weekend_nights', 'no_of_week_nights', 'type_of_meal_plan', 'required_car_parking_space', 'room_type_reserved', 'lead_time', 'arrival_year', 'arrival_month', 'arrival_date', 'market_segment_type', 'repeated_guest', 'no_of_previous_cancellations', 'no_of_previous_bookings_not_canceled', 'avg_price_per_room', 'no_of_special_requests', 'booking_status'], 'Datetime': [], 'Others': []}
Column information for the transformed test dataset:
{'Category': [], 'Numeric': ['id', 'no_of_adults', 'no_of_children', 'no_of_weekend_nights', 'no_of_week_nights', 'type_of_meal_plan', 'required_car_parking_space', 'room_type_reserved', 'lead_time', 'arrival_year', 'arrival_month', 'arrival_date', 'market_segment_type', 'repeated_guest', 'no_of_previous_cancellations', 'no_of_previous_bookings_not_canceled', 'avg_price_per_room', 'no_of_special_requests', 'booking_status'], 'D

In [3]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Split the data into features and target
X_train = train_df_copy.drop(columns=['id', 'booking_status'])
y_train = train_df_copy['booking_status']
X_test = test_df_copy.drop(columns=['id', 'booking_status'])
y_test = test_df_copy['booking_status']

# Split the training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Define the LightGBM dataset
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# Set the parameters for the LightGBM model
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0
}

# Train the LightGBM model
model = lgb.train(params, train_data, valid_sets=[val_data], num_boost_round=1000, early_stopping_rounds=100)

# Make predictions on the test set
y_pred = model.predict(X_test, num_iteration=model.best_iteration)

# Calculate the AUC-ROC score
auc_roc = roc_auc_score(y_test, y_pred)
auc_roc


TypeError: train() got an unexpected keyword argument 'early_stopping_rounds'

In [4]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Split the data into features and target
X_train = train_df_copy.drop(columns=['id', 'booking_status'])
y_train = train_df_copy['booking_status']
X_test = test_df_copy.drop(columns=['id', 'booking_status'])
y_test = test_df_copy['booking_status']

# Split the training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Define the LightGBM dataset
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# Set the parameters for the LightGBM model
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0
}

# Train the LightGBM model with early stopping
model = lgb.train(params, train_data, valid_sets=[val_data], num_boost_round=1000, callbacks=[lgb.early_stopping(stopping_rounds=100)])

# Make predictions on the test set
y_pred = model.predict(X_test, num_iteration=model.best_iteration)

# Calculate the AUC-ROC score
auc_roc = roc_auc_score(y_test, y_pred)
auc_roc

Training until validation scores don't improve for 100 rounds


Early stopping, best iteration is:
[529]	valid_0's auc: 0.892591


0.8969540096790476

In [5]:
# Since the model has already been trained and the AUC-ROC score has been calculated,
# we can directly report the AUC-ROC score from the previous execution result.
auc_roc = 0.8969540096790476
auc_roc


0.8969540096790476